In [1]:
import geopandas as gpd
import numpy as np
from pathlib import Path

import mnk.substrat as subkart
import mnk
import pandas as pd
import mnk.utils


# Snippet for small tasks

### Add additional feature layers to geoserver

In [ ]:
source = "https://storage.googleapis.com/niva-geodata/MarintNaturKart/input/kartverket/sjoekart_dybdedata_trening_norge.geo.parquet"
mnk.utils.parquet_to_postgis(source)

In [2]:
marine_vanntyper = gpd.read_parquet("gs://niva-geodata/MarintNaturKart/input/mdir/NyTypologi2022.geo.parquet").to_crs(
    "EPSG:25833"
)
fname = "nisjedata-substrat-marine-vanntyper_norge"
marine_vanntyper = subkart.features.marine_vanntyper_preprocess(marine_vanntyper)
marine_vanntyper["beskrivelse"] = marine_vanntyper['Type'].apply(lambda x: subkart.features.MARINE_VANN_TYPE_DESC[x])
mnk.utils.to_postgis(marine_vanntyper, fname)

Table nisjedata_substrat_marine_vanntyper_norge uploaded to PostGIS.


## Create features tiffs

In [2]:
res = subkart.features.RESOLUTION
nodata = 255
crs = "EPSG:25833"

In [4]:
bolge = mnk.sources.bolge_exposure()
dem_norge = mnk.sources.dem_data()

In [16]:
for region_name, region_list in mnk.sources.REGIONS.items():
    print(f"Processing region: {region_name}")
    gdf_sea_map_region = mnk.sources.sea_map_basisdata(region_list)
    gdf_points = mnk.sources.depth_point_data(region_list)
    gdf_sea_map_region = subkart.features.depth_preprocess(gdf_sea_map_region)
    transform, out_shape, bounds = subkart.features.to_raster_shapes(gdf_sea_map_region, res=res)
    dem_region = subkart.utils.resample_dem(dem_norge.crop(bounds), out_shape, transform, crs)
    bolge_region = bolge.reproject(
        crs=crs, res=res, bounds=dict(left=bounds[0], bottom=bounds[1], right=bounds[2], top=bounds[3])
    )

    X, valid_attrs, out_shape, transform = subkart.features.build(
        dem_region, gdf_sea_map_region, bolge_region, valid_mask=None, res=res, dtype=np.float32,
        gdf_points=gdf_points,
    )

    subkart.utils.save_feature_rasters(region_name, X, valid_attrs, transform, out_shape, dem_region.nodata)

Processing region: sor-ost
Preparing sea_avg_depth...


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing sea_avg_slope...


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing sea_compactness...


...

Preparing sea_compactness...


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing sea_convexity...


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing marine types...
Stacking feature arrays...


In [ ]:
for name in subkart.features.DEPTH_NAMES:
    file_list = [Path(f"features/{region_name}_{name}.tif") for region_name in mnk.sources.REGIONS.keys()]

    subkart.utils.merge_rasters(file_list, Path(f"features/{name}_merged.tif"), nodata=dem_region.nodata)

## Create Preprocessed Depth data

In [2]:
county_files = []
for f in mnk.sources.FYLKER:
    code = next((c for c in mnk.sources.FYLKER if c.endswith(f)))
    county_files.append(
        Path(f"../geonorge/Basisdata_{code}_25833_Dybdedata_Dybdeareal.geo.parquet")
    )

gdfs = [gpd.read_parquet(path) for path in county_files]

gdf_depth = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))

gdf = subkart.features.depth_preprocess(gdf_depth, is_rerun=True)

gdf.to_parquet(Path("../geonorge/sjoekart_dybdedata_trening_norge.geo.parquet"))

## Create torrfall polygons

In [ ]:
mnk.sources.prepare_torrfall()

## Simplified AOI from marine vanntyper dataset

In [2]:
mnk.sources.prepare_mv_aoi()

In [2]:
mnk.sources.prepare_land_data()

Written GeoPackage to /home/kim/work/marint-naturkart-nivaR/geonorge/Basisdata_Landareal.gpkg
